In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_3.py ──
"""
Shared infrastructure for MLFP04 Exercise 3 — Dimensionality Reduction.

Contains: data loading, scaling, common output directory, KMeans-based
silhouette evaluation in the embedding space. Technique-specific code
(PCA/KPCA/t-SNE/UMAP algorithms and their plots) lives in the per-
technique files, NOT here.

"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml import ExperimentTracker
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT + REPRODUCIBILITY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex3_dimreduce"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEFAULT_N_CLUSTERS = 4

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customers (reused from MLFP03)
# ════════════════════════════════════════════════════════════════════════


def load_customer_matrix() -> tuple[np.ndarray, list[str], pl.DataFrame]:
    """Load e-commerce customers, standardise numeric features.

    Returns:
        X          : (n_samples, n_features) standardised float matrix
        feature_cols: list of feature column names in order
        df_raw     : the raw polars DataFrame before scaling
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
        and c not in ("customer_id",)
    ]

    df_clean = customers.drop_nulls(subset=feature_cols)
    X_raw, _, _ = to_sklearn_input(df_clean, feature_columns=feature_cols)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    return X, feature_cols, df_clean


# ════════════════════════════════════════════════════════════════════════
# EMBEDDING-SPACE CLUSTER QUALITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_embedding_silhouette(
    embedding: np.ndarray,
    n_clusters: int = DEFAULT_N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> float:
    """Fit KMeans in the embedding space and return the silhouette score.

    This is the standard "does the reducer preserve structure?" probe used
    across all five technique files. Returns -1.0 when only one cluster is
    found (e.g. collapsed embedding).
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km.fit_predict(embedding)
    if len(set(labels)) < 2:
        return -1.0
    return float(silhouette_score(embedding, labels))


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — used by KPCA / t-SNE / UMAP / Isomap for kernel-cost paths
# ════════════════════════════════════════════════════════════════════════


def subsample_indices(
    n_samples: int, n_target: int, random_state: int = RANDOM_STATE
) -> np.ndarray:
    """Deterministic subsample indices for expensive O(n^2) methods."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, min(n_target, n_samples), replace=False)


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every dim-reduction technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_3 lesson logs its sweep + final-fit metrics to a single
# SQLite store so students can compare PCA / Kernel-PCA / t-SNE / UMAP
# embedding-quality runs after the lesson group ends. Mirrors the ex_1
# clustering-zoo pattern; separate DB so dim-reduction has its own
# leaderboard distinct from clustering.

DIMREDUCE_DB = "sqlite:///mlfp04_ex3_dimreduction.db"
EXPERIMENT_NAME = "m4_dimreduction_zoo"


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the dim-reduction ExperimentTracker (kailash-ml 1.5.1)."""
    tracker = await ExperimentTracker.create(store_url=DIMREDUCE_DB)
    return tracker, EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block. See ``rules/patterns.md`` § "Async Resource Cleanup".
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(name, float(value))
        if series_metrics:
            for name, values in series_metrics.items():
                for step, value in enumerate(values, start=1):
                    await run.log_metric(name, float(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 3.3: t-SNE for local structure
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Understand what t-SNE optimises (KL divergence of neighbourhoods)
#   - Tune the perplexity hyperparameter
#   - Recognise the three classic t-SNE pitfalls
#   - Know when t-SNE is a visualisation tool, not a feature extractor
#
# PREREQUISITES: 01_pca.py — we pre-reduce with PCA before t-SNE.
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — t-SNE as a neighbourhood-preserving map
#   2. Build — PCA pre-reduction + t-SNE at 4 perplexity values
#   3. Train — KL divergence + silhouette per perplexity
#   4. Visualise — 2D embedding scatter + perplexity comparison
#   5. Apply — Changi Airport passenger journey clustering
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from kailash_ml import ModelVisualizer



### Kailash-ML ExperimentTracker — every dim-reduction run logs here


In [ ]:
tracker, exp_name = setup_engines()



## THEORY — t-SNE in one paragraph


In [ ]:
# Build a high-dim Gaussian P over pairwise neighbours, a low-dim
# Student-t Q, then minimise KL(P || Q). Nearby points stay close in 2D.
# PERPLEXITY = effective # neighbours (5..50). PITFALLS:
#   A. Cluster SIZES are meaningless.
#   B. Inter-cluster distances are meaningless.
#   C. No out-of-sample transform — t-SNE is a picture, not a feature.



## TASK 2 — BUILD: PCA pre-reduction + subsample


In [ ]:
X, feature_cols, _ = load_customer_matrix()
n_samples, n_features = X.shape

# TODO: pre-reduce X with PCA to min(10, n_features) components. This is
# standard practice before t-SNE — it denoises without losing distances.
# Hint: PCA(n_components=..., random_state=42)
pca_pre = ____
X_pca = ____

# TODO: subsample 3,000 rows for t-SNE (Barnes-Hut scales but has a big
# constant factor). Use the shared helper.
idx = ____
X_tsne_input = X_pca[idx]
print(f"=== t-SNE input ===  n={X_tsne_input.shape[0]:,}, d={X_tsne_input.shape[1]}")



## TASK 3 — TRAIN: sweep perplexity


In [ ]:
tsne_results: dict[int, dict] = {}
perplexities = [5, 15, 30, 50]

print(f"\n=== t-SNE perplexity sweep ===")
print(f"{'perplexity':>12}{'KL div':>14}{'silhouette':>14}{'time (s)':>12}")
print("-" * 52)

for perplexity in perplexities:
    t0 = time.time()
    # TODO: build a TSNE with n_components=2, the current perplexity,
    # max_iter=1000, random_state=42, init='pca', learning_rate='auto'.
    tsne = ____
    # TODO: fit_transform the pre-reduced input.
    embedding = ____
    elapsed = time.time() - t0

    # TODO: score the 2D embedding with the shared silhouette helper.
    sil = ____
    tsne_results[perplexity] = {
        "embedding": embedding,
        "kl": float(tsne.kl_divergence_),
        "silhouette": sil,
        "time_s": elapsed,
    }
    print(f"{perplexity:>12}{tsne.kl_divergence_:>14.4f}{sil:>14.4f}{elapsed:>11.1f}")



### Checkpoint 1


In [ ]:
assert len(tsne_results) == 4, "Must test 4 perplexity values"
for perp, res in tsne_results.items():
    assert res["embedding"].shape[1] == 2, "t-SNE must produce 2D output"
    assert res["kl"] > 0, "KL divergence must be positive"
print("\n[ok] Checkpoint 1 — 2D embeddings across 4 perplexity settings")



## TASK 4 — VISUALISE: perplexity comparison


In [ ]:
viz = ModelVisualizer()
fig_perp = viz.metric_comparison(
    {
        f"perplexity={p}": {"Silhouette": r["silhouette"], "KL": r["kl"]}
        for p, r in tsne_results.items()
    }
)
fig_perp.update_layout(title="t-SNE: perplexity vs KL divergence and silhouette")
perp_path = OUTPUT_DIR / "03_tsne_perplexity.html"
fig_perp.write_html(str(perp_path))
print(f"\nSaved: {perp_path}")

print("\nPerplexity guidance:")
print("  5  — micro-clusters (fragile)")
print("  15 — fine local structure")
print("  30 — balanced default")
print("  50 — smoother, fewer isolated clusters")



## TASK 5 — APPLY: Changi Airport passenger journey clustering


In [ ]:
# SCENARIO: CAG instruments T3 with 80+ touchpoints per passenger. The
# retail team wants MICRO-SEGMENTS inside the transit-passenger group:
# families, business travellers, premium cabins, budget lingerers. t-SNE
# captures LOCAL structure — perfect for a dashboard the retail planners
# can act on. Perplexity is tuned until the cluster granularity matches
# the 9 retail managers on duty. 7% F&B uplift on ~S$280M GMV = ~S$19.6M/yr.
# NEVER feed t-SNE coordinates into a downstream model — stochastic.

best_p, best_r = max(tsne_results.items(), key=lambda kv: kv[1]["silhouette"])
print(f"\n=== Changi-style micro-segment projection ===")
print(f"  Best perplexity : {best_p}")
print(f"  Silhouette      : {best_r['silhouette']:.4f}")
print(f"  KL divergence   : {best_r['kl']:.4f}")



## TRACK — Log this lesson's run to the kailash-ml ExperimentTracker


In [ ]:
# Per-perplexity KL/silhouette/time scalars + parallel series go into the
# m4_dimreduction_zoo experiment.

perplexities_sorted = sorted(tsne_results.keys())

# TODO: call track_run with run_name f"tsne_perp_{best_p}". scalar_metrics
# headline is best_silhouette + best_kl, then |-merge three per-perplexity
# dicts (silhouette, kl, time_s) over tsne_results.items(). series_metrics:
# parallel silhouette + KL arrays in perplexities_sorted order.
track_run(
    tracker,
    exp_name,
    run_name=____,
    params={
        "algorithm": "tsne",
        "n_components": 2,
        "n_subsample": int(X_tsne_input.shape[0]),
        "pca_pre_components": int(X_tsne_input.shape[1]),
        "perplexities": ",".join(str(p) for p in perplexities_sorted),
        "best_perplexity": best_p,
    },
    scalar_metrics={
        "best_silhouette": float(best_r["silhouette"]),
        "best_kl": float(best_r["kl"]),
    }
    | ____
    | ____
    | ____,
    series_metrics={
        "sweep_silhouette": ____,
        "sweep_kl": ____,
    },
)
print(f"  [tracked] perplexity sweep logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — DimReductionEngine.reduce(algorithm='tsne')


In [ ]:
# kailash-ml 1.5.1's DimReductionEngine wraps sklearn t-SNE under the same
# `reduce` surface that backed PCA in lesson 01. The engine handles the
# polars→numpy conversion, runs t-SNE, returns a DimReductionResult — one
# sync call.

import polars as pl

from kailash_ml.engines.dim_reduction import DimReductionEngine

# Engine takes raw features (not the PCA-pre-reduced matrix) and runs the
# whole pipeline; we slice down to the same subsample for fairness.
sub_idx = idx
cust_df = pl.from_numpy(X[sub_idx], schema=feature_cols)

# TODO: instantiate DimReductionEngine and call .reduce on cust_df with
# algorithm='tsne', n_components=2, perplexity=best_p.
dimreduce = ____
reduce_result = ____
print(
    f"  DimReductionEngine.reduce(tsne, perplexity={best_p}): "
    f"embedding shape=({len(reduce_result.transformed)}, "
    f"{reduce_result.n_components})  "
    f"kl={reduce_result.metrics.get('kl_divergence', float('nan')):.4f}"
)
print()
print("  Same t-SNE you swept by hand — wrapped under the engine surface")
print("  that backs pca / tsne / umap / nmf. The leaderboard now compares")
print("  this perplexity with the PCA baseline from lesson 01.\n")



## REFLECTION


[x] Ran t-SNE at 4 perplexity values and measured KL + silhouette
  [x] Pre-reduced with PCA before t-SNE (standard practice)
  [x] Recognised the three pitfalls: cluster size, inter-cluster
      distance, no out-of-sample transform
  [x] Sized t-SNE for a Changi retail dashboard where the output is a
      visual, not a feature

  KEY INSIGHT: t-SNE is a PICTURE generator. When your deliverable is
  an insight for a human, t-SNE is brilliant. When it's a feature for
  another model, use PCA or UMAP.

  Next: 04_umap.py adds out-of-sample transform.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)


# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

